In [0]:
%run ../utils/utils

In [0]:
%run ../config/config

In [0]:
%run ../ingestion/bronze

In [0]:
%run ../ingestion/silver


In [0]:
%run ../ingestion/gold

In [0]:
%run ../ingestion/insights

In [0]:
# Databricks notebook source
# MAGIC %run ../utils/utils

# COMMAND ----------

import pyspark.sql.functions as F

TABELA_ALVO = "ecommerce_enderecos"

df_silver = ler_delta("silver", TABELA_ALVO, STORAGE_OPTIONS).select("id_endereco").distinct()
df_quarentena = ler_delta("silver/quarentena", TABELA_ALVO, STORAGE_OPTIONS).select("id_endereco").distinct()

qtd_silver = df_silver.count()
qtd_quarentena = df_quarentena.count()

df_sobreposicao = df_silver.join(df_quarentena, "id_endereco", "inner")
qtd_sobreposicao = df_sobreposicao.count()

print(f"IDs distintos na Silver:      {qtd_silver}")
print(f"IDs distintos na Quarentena:  {qtd_quarentena}")
print(f"IDs presentes em AMBAS:       {qtd_sobreposicao}")

if qtd_sobreposicao > 0:
    print("\nCONFIRMADO: existem IDs simultaneamente válidos e quarentenados.")
    print("Isso só é possível se esses IDs foram processados em execuções")
    print("diferentes, com resets físicos parciais entre elas.")
    display(df_sobreposicao.limit(30))
else:
    print("\nNenhuma sobreposição encontrada — Silver e Quarentena são mutuamente exclusivas.")